# TopicBank: Bank Creation Experiment

Here we are going to collect interpretable topics (automatically, using topic coherence) from multiple model training.
These topics constitute *topic bank*.
And then the topic bank is going to be used for estimating topic models quality in the notebook [TopicBank-Experiment: Model Validation](TopicBank-Experiment-ModelValidation.ipynb).

The process is repeated for several datasets (some of them are already downloadable using [TopicNet](https://github.com/machine-intelligence-laboratory/TopicNet) library).

# Contents<a id="contents"></a>

* [Data](#data)
    * [Coocs](#coocs)
        * [Lower Memory Consumption (or a Bit of Shamanism. Part 1)](#optimizing-memory)
    * [Documents for Coherence Scores](#docs-for-cohs)
        * [Lower Time Consumption in Case of Big Datasets (or a Bit of Shamanism. Part 2)](#optimizing-time)
* [Experiment](#experiment)
    * [Scores](#scores)
    * [Bank Creation](#bank-creation)
* [Postprocessing](#postprocessing)

In [1]:
# General imports

import dill
import itertools
import json
import numpy as np
import os
import pandas as pd
import sys

from enum import Enum
from scipy.stats import gaussian_kde
from matplotlib import pyplot as plt
from tqdm import tqdm
from typing import (
    Dict,
    Iterable,
)

%matplotlib inline

In [2]:
# Making `topnum` module visible for Python

sys.path.insert(0, '../OptimalNumberOfTopics')

In [3]:
# Optimal number of topics

from topicnet.cooking_machine import Dataset

from topnum.data.vowpal_wabbit_text_collection import VowpalWabbitTextCollection
from topnum.scores import (
    PerplexityScore,
    SparsityPhiScore,
    SparsityThetaScore,
)
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.scores._base_coherence_score import (
    SpecificityEstimationMethod,
    TextType,
    WordTopicRelatednessType,
)
from topnum.scores.intratext_coherence_score import ComputationMethod
from topnum.search_methods import TopicBankMethod
from topnum.search_methods.topic_bank.topic_bank import TopicBank
from topnum.search_methods.topic_bank.one_model_train_funcs import (
    default_train_func,

    # Functions below are not used (but could have been)

#     regularization_train_func,
#     specific_initial_phi_train_func,
#     background_topics_train_func,

)

## Data<a id="data"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Loading data from disk, creating batches, dictionary, gathering cooccurrence statistics...

In [4]:
DATA_FOLDER_PATH = '/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager'

In [5]:
sorted(os.listdir(DATA_FOLDER_PATH))

['20NG.csv',
 '20NG__internals',
 'Brown',
 'Brown_BOW.csv',
 'Brown_NOOW.csv',
 'MKB10.csv',
 'MKB10__internals',
 'RTL_Wiki.csv',
 'RTL_Wiki_person.csv',
 'RTL_Wiki_person__internals',
 'Reuters',
 'Reuters_BOW.csv',
 'Reuters_NOOW.csv',
 'WikiRef-220',
 '__init__.py',
 '__pycache__',
 'api.py',
 'postnauka.csv',
 'postnauka__internals',
 'ruwiki_good.txt',
 'ruwiki_good__internals',
 'wiki_ref220_bow.csv',
 'wiki_ref220_natural_order.csv']

In [6]:
class DatasetName(Enum):
    POSTNAUKA = 'Post_Science'
    # REUTERS = 'Reuters'
    # BROWN = 'Brown'
    TWENTY_NEWSGROUPS = '20_Newsgroups'
    GOOD_RU_WIKI = 'Good_RU_Wiki'
    MKB10 = 'MKB_10'

In [7]:
DATASET_NAME_TO_DATASET_FILE_PATH = {
    DatasetName.POSTNAUKA: os.path.join(
        DATA_FOLDER_PATH, 'postnauka.csv'
    ),
    # DatasetName.REUTERS: os.path.join(
    #     DATA_FOLDER_PATH, 'Reuters.csv'
    # ),
    # DatasetName.BROWN: os.path.join(
    #     DATA_FOLDER_PATH, 'Brown.csv'
    # ),
    DatasetName.TWENTY_NEWSGROUPS: os.path.join(
        DATA_FOLDER_PATH, '20NG.csv'
    ),
    # DatasetName.AG_NEWS: os.path.join(
    #     DATA_FOLDER_PATH, 'AG_News.csv'
    # ),
    # DatasetName.WATAN: os.path.join(
    #     DATA_FOLDER_PATH, 'Watan2004.csv'
    # ),
    # DatasetName.HABRAHABR: os.path.join(
    #     DATA_FOLDER_PATH, 'Habrahabr.csv'
    # ),
    DatasetName.GOOD_RU_WIKI: os.path.join(
        DATA_FOLDER_PATH, 'ruwiki_good.txt'
    ),
    DatasetName.MKB10: os.path.join(
        DATA_FOLDER_PATH, 'MKB10.csv'
    ),
}

In [8]:
DATASET_NAME = DatasetName.MKB10  # select a dataset here

DATASET_FILE_PATH = DATASET_NAME_TO_DATASET_FILE_PATH[DATASET_NAME]

Checking if all OK with data, what modalities does the collection have.

In [9]:
! head -n 2 $DATASET_FILE_PATH

id,raw_text,vw_text
«Бедная_симптомами»_шизофрения,"«Бе́дная симпто́мами» шизофрени́я — подтип шизотипического расстройства в российской версии МКБ-10[1] (ранее считавшийся «простым вариантом вялопротекающей шизофрении»[2][3] и «первичным дефект-психозом»[4][3]), проявляющийся преимущественно негативными симптомами (апатией, астеническим дефектом, суженным или уплощённым аффектом, социальной аутизацией, но без бреда и галлюцинаций).


In [10]:
def get_dataset_internals_folder_path(dataset_name: DatasetName) -> str:
    return os.path.join('.', dataset_name.value + '__internals')

In [11]:
DATASET_INTERNALS_FOLDER_PATH = get_dataset_internals_folder_path(DATASET_NAME)

In [12]:
DATASET_INTERNALS_FOLDER_PATH

'./MKB_10__internals'

In [13]:
%%time

# If using really big datasets (like Habrahabr),
# one may need to set this equal `False`
KEEP_DATASET_IN_MEMORY = True

DATASET = Dataset(
    DATASET_FILE_PATH,
    internals_folder_path=DATASET_INTERNALS_FOLDER_PATH,
    keep_in_memory=KEEP_DATASET_IN_MEMORY,
)

CPU times: user 1.51 s, sys: 149 ms, total: 1.66 s
Wall time: 1.59 s


Looking what is inside dataset's folder

In [14]:
os.listdir(DATASET_INTERNALS_FOLDER_PATH)

['vw.txt', 'result2', 'batches', 'dict.dict', 'result']

Creating batches

In [15]:
DATASET.get_batch_vectorizer()

artm.BatchVectorizer(data_path="./MKB_10__internals/batches", num_batches=3)

In [16]:
os.listdir(DATASET_INTERNALS_FOLDER_PATH)

['vw.txt', 'result2', 'batches', 'dict.dict', 'result']

In [17]:
if KEEP_DATASET_IN_MEMORY:
    DOCUMENTS = list(DATASET._data.index)
else:
    DOCUMENTS = list(DATASET._data_index)

NUM_DOCUMENTS = len(DOCUMENTS)

print(f'Num documents: {NUM_DOCUMENTS}')

Num documents: 2036


Let's look at some text samples

In [18]:
DATASET._data.head()

,id,raw_text,vw_text
id,,,
«Бедная_симптомами»_шизофрения,«Бедная_симптомами»_шизофрения,«Бе́дная симпто́мами» шизофрени́я — подтип шиз...,«Бедная_симптомами»_шизофрения |@text бедный с...
"46,XX/46,XY","46,XX/46,XY","46,XX/46,XY (тетрагаметный химеризм) — это раз...","46,XX/46,XY |@text <person> химеризм разновидн..."
"Синдром_48,_XXXY","Синдром_48,_XXXY","Синдром 48, XXXY — это генетическое состояние,...","Синдром_48,_XXXY |@text синдром xxxy генетичес..."
"Синдром_48,_XXYY","Синдром_48,_XXYY","Синдром 48, XXYY — это аномалия хромосом, при ...","Синдром_48,_XXYY |@text синдром xxyy аномалия ..."
"Синдром_48,_XYYY","Синдром_48,_XYYY","Синдром 48, XYYY — чрезвычайно редкая анеуплои...","Синдром_48,_XYYY |@text синдром xyyy чрезвычаи..."


In [19]:
DATASET.get_possible_modalities()

{'@letter', '@ngram', '@text'}

In [20]:
MAIN_MODALITY = '@text'

In [21]:
DATASET.get_dictionary()

artm.Dictionary(name=a536dcf1-32ce-4d6e-93c2-0994face6039, num_entries=244551)

In [22]:
dictionary = DATASET.get_dictionary()

In [23]:
print(dictionary)

for modality in DATASET.get_possible_modalities():
    if modality not in [MAIN_MODALITY]:
        dictionary.filter(class_id=modality, max_df=0, inplace=True)

artm.Dictionary(name=a536dcf1-32ce-4d6e-93c2-0994face6039, num_entries=244551)


In [24]:
dictionary.filter(min_df=2, max_df_rate=0.5)

artm.Dictionary(name=a536dcf1-32ce-4d6e-93c2-0994face6039, num_entries=22608)

In [25]:
DATASET._cached_dict = dictionary

In [26]:
DATASET.get_dictionary()

artm.Dictionary(name=a536dcf1-32ce-4d6e-93c2-0994face6039, num_entries=22608)

In [27]:
import scipy

from typing import List

from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)

In [28]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices=None,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        T, W = phi.shape
        # T = len(topic_indices)
        topic_indices = list(range(T))

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            # print(top, phi.shape, doc_co_occurrences.shape)
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [29]:
%%time

occurences, co_occurences = calc_doc_occurrences(DATASET, MAIN_MODALITY)

CPU times: user 3.86 s, sys: 115 ms, total: 3.98 s
Wall time: 3.93 s


In [30]:
co_occurences.shape

(22608, 22608)

In [31]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    DATASET.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [32]:
import copy


class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    @property
    def name(self):
        return self._name

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

    def compute(
            self,
            model,
            topics: List[str] = None,
            documents: List[str] = None) -> Dict[str, float]:

        values = self.call_by_topic(model)

        phi = model.get_phi()

        if topics is None:
            topics = list(phi.columns)

            if hasattr(model, 'has_bcg'):
                print(f'Detected bcg topics! Skipping for coherence computation (and will have {len(topics) - 1} topics).')

                topics = topics[:-1]
        else:
            assert False

        index2topic = {phi.columns.get_loc(t): t for t in topics}
        topic2index = {t: i for i, t in index2topic.items()}

        if hasattr(model, 'has_bcg'):
            assert list(index2topic.keys()) == list(values.keys())[:-1]
        else:
            assert list(index2topic.keys()) == list(values.keys())

        result = {
            t: float(values[topic2index[t]])
            for t in topics
        }

        assert len(result) == len(index2topic)

        return result

    def _attach(self, model: TopicModel):
        if self._name in model.custom_scores:
            print(
                f'Score with such name "{self._name}" already attached to model!'
                f' So rewriting it...'
                f' All model\'s custom scores: {list(model.custom_scores.keys())}'
            )

        # TODO: TopicModel should provide ability to add custom scores
        model.custom_scores[self.name] = copy.deepcopy(self)

## Experiment<a id="experiment"></a>

Finally we are getting to the main part!)

### Scores (for Topics and Models)<a id="scores"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Here we define a lot of scores (which mainly differ in initial parameters).

In [36]:
ONE_MODEL_NUM_TOPICS = 50
NUM_TOP_WORDS = 20

In [37]:
top = NUM_TOP_WORDS
target_topic_indices = list(range(ONE_MODEL_NUM_TOPICS))

coherence_score = TopTokenCoherence(
    name=f'coherence_{top}',
    func=create_pmi_top_function(
        occurences, co_occurences,
        DATASET.get_dataset().shape[0], [top],
        # topic_indices=target_topic_indices,
        co_occurrences_smooth=1e-2,
    )
)

diversity_scores = [
    DiversityScore(
        name=f'diversity_{metric}',
        metric=metric,
        class_ids=MAIN_MODALITY,
    )

    for metric in KNOWN_METRICS
]

Other coherence score variations

And a pair of default ARTM scores (these ones are fast)

In [38]:
other_scores = [
    PerplexityScore(
        name='perplexity'
    ),
]

### Bank Creation<a id="bank-creation"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Here we finally run the experiment!

In [39]:
NUM_ITERATIONS = 20

In [40]:
seed = 0

In [41]:
# We use only one train function here
# Other variations are also possible
# It would be even better to make bank using several train functions
# However, it would also take way more time 

TRAIN_FUNCS = default_train_func  # default train func

In [42]:
DATASET_INTERNALS_FOLDER_PATH

'./MKB_10__internals'

In [43]:
SEARCH_RESULTS_FOLDER_PATH = os.path.join(
    DATASET_INTERNALS_FOLDER_PATH, 'result_50'
)

# File with some info about the process
SEARCH_RESULT_FILE_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'search_result__{seed}.json'
)

# Bank, with topics and their score values
BANK_FOLDER_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'bank__{seed}'
)

In [44]:
! echo $DATASET_INTERNALS_FOLDER_PATH
! ls -alh $DATASET_INTERNALS_FOLDER_PATH

./MKB_10__internals
total 50M
drwxrwxr-x  5 alekseev_v mil_lab 4,0K мар 28 09:01 .
drwxrwxr-x 11 alekseev_v mil_lab 4,0K мар 28 09:01 ..
drwxrwxr-x  2 alekseev_v mil_lab 4,0K мар 26 18:06 batches
-rw-rw-r--  1 alekseev_v mil_lab  14M мар 26 18:06 dict.dict
drwxrwxr-x  3 alekseev_v mil_lab 4,0K мар 26 18:07 result
drwxrwxr-x  3 alekseev_v mil_lab 4,0K мар 26 18:40 result2
-rw-rw-r--  1 alekseev_v mil_lab  36M мар 26 18:06 vw.txt


In [45]:
SEARCH_RESULTS_FOLDER_PATH

'./MKB_10__internals/result_50'

In [46]:
! ls $SEARCH_RESULTS_FOLDER_PATH

ls: cannot access './MKB_10__internals/result_50': No such file or directory


In [47]:
BANK_FOLDER_PATH

'./MKB_10__internals/result_50/bank__0'

In [48]:
os.makedirs(SEARCH_RESULTS_FOLDER_PATH, exist_ok=True)
os.makedirs(BANK_FOLDER_PATH, exist_ok=True)

In [49]:
seed

0

One cay vary some parameters below (for example `max_num_models` and `num_fit_iterations`).

In [50]:
optimizer = TopicBankMethod(
    data        = DATASET,
    main_modality = MAIN_MODALITY,
    
    min_df_rate = 0.0,  # dictionary filtering has already been done little earlier
    max_df_rate = 1.0,  #   so we don't want these parameters to have any effect

    main_topic_score   = coherence_score,
    other_topic_scores = [],
    other_scores       = [coherence_score] + diversity_scores + other_scores,
   # documents          = TEST_DOCUMENTS,

    start_model_number   = 0,
    max_num_models       = 20,
    one_model_num_topics = ONE_MODEL_NUM_TOPICS,  # 100,
    num_fit_iterations   = NUM_ITERATIONS,  # 100,  # 100 should be enough;
                                 # however, for big data better to reduce this one
                                 # (otherwise the process will be too slow)

    topic_score_threshold_percentile = 90,

    save_bank         = True,
    save_model_topics = True,
    save_file_path    = SEARCH_RESULT_FILE_PATH,
    bank_folder_path  = BANK_FOLDER_PATH,

    train_funcs = TRAIN_FUNCS,
    
    verbose = True,
)

# TODO: use Holdout Perplexity as Stop score

In [51]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

Checking file paths

In [52]:
! echo $DATASET_INTERNALS_FOLDER_PATH
! ls $DATASET_INTERNALS_FOLDER_PATH

./MKB_10__internals
batches  dict.dict  result  result2  result_50	vw.txt


In [53]:
optimizer._save_file_path

'./MKB_10__internals/result_50/search_result__0.json'

In [54]:
optimizer._topic_bank._path

'./MKB_10__internals/result_50/bank__0'

Fulfilling the search (get ready for a really long process!):

In [55]:
%%time

optimizer.search_for_optimum(DATASET)

100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.69it/s]


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 16209.0234375, 'coherence_20': 1.0179465178051823, 'diversity_euclidean': 0.07064976952559238, 'diversity_jensenshannon': 0.6415289643204914, 'diversity_hellinger': 0.7493474373263616, 'diversity_cosine': 0.8021132223986681, 'perplexity': 16209.0234375, 'ppl_fair': 16209.0234375, 'ppl_cheatty': 3895.4580078125}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.28it/s]
Creating first level with 5 topics. Dictionary: artm.Dictionary(name=a536dcf1-32ce-4d6e-93c2-0994face6039, num_entries=22608).
Copying phi for the first level. Phi shape: (22608, 5). First words: MultiIndex([('@text',              'cdk'),
            ('@text',    'внутриядерный'),
            ('@text',      'соотнесение'),
            ('@text',    'ангиоматозный'),
            ('@text',    'хориоменингит'),
            ('@text',    'герминативный'),
            ('@text',            'пенни'),
            ('@text',         'мозамбик'),
            ('@text',       

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 11601.009765625, 'coherence_20': 1.023528517974686, 'diversity_euclidean': 0.07445770158910985, 'diversity_jensenshannon': 0.6559229223663022, 'diversity_hellinger': 0.7668578313614962, 'diversity_cosine': 0.8337669450860792, 'perplexity': 11601.009765625, 'ppl_fair': 11601.009765625, 'ppl_cheatty': 3714.803466796875}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.08it/s]
Creating first level with 6 topics. Dictionary: artm.Dictionary(name=a536dcf1-32ce-4d6e-93c2-0994face6039, num_entries=22608).
Copying phi for the first level. Phi shape: (22608, 6). First words: MultiIndex([('@text',              'cdk'),
            ('@text',    'внутриядерный'),
            ('@text',      'соотнесение'),
            ('@text',    'ангиоматозный'),
            ('@text',    'хориоменингит'),
            ('@text',    'герминативный'),
            ('@text',            'пенни'),
            ('@text',         'мозамбик'),
            ('@text',

...

Bank scores: {'perplexity_score': 6195.87060546875, 'coherence_20': 1.0411019210688377, 'diversity_euclidean': 0.07473547588172519, 'diversity_jensenshannon': 0.6485777195082487, 'diversity_hellinger': 0.7571550974181261, 'diversity_cosine': 0.8414739064853655, 'perplexity': 6195.87060546875, 'ppl_fair': 6195.87060546875, 'ppl_cheatty': 3302.781982421875}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.62it/s]
Creating first level with 11 topics. Dictionary: artm.Dictionary(name=a536dcf1-32ce-4d6e-93c2-0994face6039, num_entries=22608).
Copying phi for the first level. Phi shape: (22608, 11). First words: MultiIndex([('@text',              'cdk'),
            ('@text',    'внутриядерный'),
            ('@text',      'соотнесение'),
            ('@text',    'ангиоматозный'),
            ('@text',    'хориоменингит'),
            ('@text',    'герминативный'),
            ('@text',            'пенни'),
            ('@text',         'мозамбик'),
            ('@

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 6195.87060546875, 'coherence_20': 1.0411019210688377, 'diversity_euclidean': 0.0747354758817252, 'diversity_jensenshannon': 0.6485777195082519, 'diversity_hellinger': 0.7571550974181251, 'diversity_cosine': 0.8414739064853652, 'perplexity': 6195.87060546875, 'ppl_fair': 6195.87060546875, 'ppl_cheatty': 3302.781982421875}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.97it/s]
Creating first level with 11 topics. Dictionary: artm.Dictionary(name=a536dcf1-32ce-4d6e-93c2-0994face6039, num_entries=22608).
Copying phi for the first level. Phi shape: (22608, 11). First words: MultiIndex([('@text',              'cdk'),
            ('@text',    'внутриядерный'),
            ('@text',      'соотнесение'),
            ('@text',    'ангиоматозный'),
            ('@text',    'хориоменингит'),
            ('@text',    'герминативный'),
            ('@text',            'пенни'),
            ('@text',         'мозамбик'),
            ('@t

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 6195.87060546875, 'coherence_20': 1.0411019210688377, 'diversity_euclidean': 0.0747354758817252, 'diversity_jensenshannon': 0.6485777195082517, 'diversity_hellinger': 0.7571550974181199, 'diversity_cosine': 0.8414739064853652, 'perplexity': 6195.87060546875, 'ppl_fair': 6195.87060546875, 'ppl_cheatty': 3302.781982421875}
100%|███████████████████████████████████████████████| 20/20 [21:16<00:00, 63.84s/it]
CPU times: user 32min 28s, sys: 44.5 s, total: 33min 12s
Wall time: 21min 16s


What topics we have in bank

In [84]:
optimizer._topic_bank.view_topics().head()

topic_0  topic_1  topic_2  topic_3  topic_4  topic_5  \
@text cdk                0.0      0.0      0.0      0.0      0.0      0.0   
      внутриядерный      0.0      0.0      0.0      0.0      0.0      0.0   
      соотнесение        0.0      0.0      0.0      0.0      0.0      0.0   
      ангиоматозный      0.0      0.0      0.0      0.0      0.0      0.0   
      хориоменингит      0.0      0.0      0.0      0.0      0.0      0.0   

                     topic_6  topic_7  topic_8  topic_9  topic_10  topic_11  \
@text cdk                0.0      0.0      0.0      0.0       0.0       0.0   
      внутриядерный      0.0      0.0      0.0      0.0       0.0       0.0   
      соотнесение        0.0      0.0      0.0      0.0       0.0       0.0   
      ангиоматозный      0.0      0.0      0.0      0.0       0.0       0.0   
      хориоменингит      0.0      0.0      0.0      0.0       0.0       0.0   

                     topic_12  topic_13  
@text cdk                 0.0       0.0  
      внутриядерный       0.0       0.0  
      соотнесение         0.0       0.0  
      ангиоматозный       0.0       0.0  
      хориоменингит       0.0       0.0

In [85]:
bank_topics = optimizer._topic_bank.view_topics()

In [86]:
bank_topics.shape

(22608, 14)

In [87]:
bank_topics.head()

topic_0  topic_1  topic_2  topic_3  topic_4  topic_5  \
@text cdk                0.0      0.0      0.0      0.0      0.0      0.0   
      внутриядерный      0.0      0.0      0.0      0.0      0.0      0.0   
      соотнесение        0.0      0.0      0.0      0.0      0.0      0.0   
      ангиоматозный      0.0      0.0      0.0      0.0      0.0      0.0   
      хориоменингит      0.0      0.0      0.0      0.0      0.0      0.0   

                     topic_6  topic_7  topic_8  topic_9  topic_10  topic_11  \
@text cdk                0.0      0.0      0.0      0.0       0.0       0.0   
      внутриядерный      0.0      0.0      0.0      0.0       0.0       0.0   
      соотнесение        0.0      0.0      0.0      0.0       0.0       0.0   
      ангиоматозный      0.0      0.0      0.0      0.0       0.0       0.0   
      хориоменингит      0.0      0.0      0.0      0.0       0.0       0.0   

                     topic_12  topic_13  
@text cdk                 0.0       0.0  
      внутриядерный       0.0       0.0  
      соотнесение         0.0       0.0  
      ангиоматозный       0.0       0.0  
      хориоменингит       0.0       0.0

In [88]:
bank_topics['topic_5'].sort_values(ascending=False)[:20]

@text  печень         0.039570
       кровь          0.014464
       желчный        0.011767
       желтуха        0.008962
       эритроцит      0.007931
       анемия         0.007641
       цирроз         0.007230
       больной        0.006944
       печеночный     0.006693
       колит          0.006646
       билирубин      0.006036
       болезнь        0.005616
       увеличение     0.005514
       нарушение      0.005498
       желчь          0.005430
       пациент        0.005394
       селезенка      0.005386
       гемоглобин     0.005211
       хронический    0.005112
       наблюдаться    0.004678
Name: topic_5, dtype: float64

And topic scores

In [89]:
optimizer._topic_bank.view_topic_scores()

,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,topic_10,topic_11,topic_12,topic_13
kernel_size,2154.000000,1925.000000,2330.000000,1745.000000,1882.000000,1964.000000,2067.000000,1722.000000,2626.000000,2594.000000,2347.000000,2184.000000,2159.000000,2091.000000
coherence_20,1.113181,1.199071,1.008168,1.581345,1.030367,0.900842,1.254779,0.969312,0.956957,1.096151,0.941750,1.077395,1.222141,0.965841
distance_to_nearest,0.000000,0.912625,0.790161,0.862318,0.817064,0.830169,0.831680,0.653847,0.823542,0.733325,0.827345,0.832594,0.831554,0.768941


All models are also saved (topics as $\Phi$ matrices and topic score values)

In [90]:
! ls $optimizer._topic_bank._path

model_0__phi.bin	    model_19__topic_scores.bin
model_0__topic_scores.bin   model_1__phi.bin
model_10__phi.bin	    model_1__topic_scores.bin
model_10__topic_scores.bin  model_2__phi.bin
model_11__phi.bin	    model_2__topic_scores.bin
model_11__topic_scores.bin  model_3__phi.bin
model_12__phi.bin	    model_3__topic_scores.bin
model_12__topic_scores.bin  model_4__phi.bin
model_13__phi.bin	    model_4__topic_scores.bin
model_13__topic_scores.bin  model_5__phi.bin
model_14__phi.bin	    model_5__topic_scores.bin
model_14__topic_scores.bin  model_6__phi.bin
model_15__phi.bin	    model_6__topic_scores.bin
model_15__topic_scores.bin  model_7__phi.bin
model_16__phi.bin	    model_7__topic_scores.bin
model_16__topic_scores.bin  model_8__phi.bin
model_17__phi.bin	    model_8__topic_scores.bin
model_17__topic_scores.bin  model_9__phi.bin
model_18__phi.bin	    model_9__topic_scores.bin
model_18__topic_scores.bin  topics.bin
model_19__phi.bin	    topic_scores.bin


In [91]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

In [92]:
optimizer._result['num_bank_topics']

[11,
 11,
 12,
 12,
 13,
 14,
 14,
 14,
 14,
 14,
 14,
 14,
 14,
 14,
 14,
 14,
 14,
 14,
 14,
 14]

In [93]:
len(optimizer._result['bank_topic_scores'])

20

In [94]:
optimizer._result['bank_scores'][-1]

{'perplexity_score': 7365.70166015625,
 'coherence_20': 1.0940928784569097,
 'diversity_euclidean': 0.08597331388915083,
 'diversity_jensenshannon': 0.6795176507276979,
 'diversity_hellinger': 0.797904102369454,
 'diversity_cosine': 0.8667473333046445,
 'perplexity': 7365.70166015625,
 'ppl_fair': 7365.70166015625,
 'ppl_cheatty': 3324.31787109375}

In [67]:
import artm
from topnum.model_constructor import KnownModel, init_plsa
from topicnet.cooking_machine.rel_toolbox_lite import (
    count_vocab_size,
    transform_regularizer,
)
from topicnet.cooking_machine.model_constructor import (
    add_standard_scores,
    create_default_topics,
    init_model,
)

def init_model_from_family(
        family: str or KnownModel,
        dataset: Dataset,
        main_modality: str,
        num_topics: int,
        seed: int,
        specific_topic_names = None,
        modalities_to_use: List[str] = None,
        num_processors: int = 3,
        model_params: dict = None,
):
    """
    Returns
    -------
    model: TopicModel() instance
    """
    if isinstance(family, KnownModel):
        family = family.value

    if modalities_to_use is None:
        modalities_to_use = [main_modality]

    custom_regs = {}

    if family == "LDA":
        model = init_lda(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "PLSA":
        model = init_plsa(
            dataset, modalities_to_use, main_modality, num_topics
        )
    elif family == "TARTM":
        model, custom_regs = init_thetaless(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "sparse":
        model = init_bcg_sparse_model(
            dataset, modalities_to_use, main_modality, num_topics, 1, model_params
        )
    elif family == "decorrelation":
        model = init_decorrelated_plsa(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "ARTM":
        model = init_baseline_artm(
            dataset, modalities_to_use, main_modality, num_topics, 1, specific_topic_names, model_params
        )
    else:
        raise ValueError(f'family: {family}')

    model.num_processors = num_processors

    if seed is not None:
        model.seed = seed

    dictionary = dataset.get_dictionary()

    # TODO: maybe this cycle is not necessary
    for modality in dataset.get_possible_modalities():
        if modality not in modalities_to_use:
            dictionary.filter(class_id=modality, max_df=0, inplace=True)

    model.initialize(dictionary)
    add_standard_scores(model, dictionary, main_modality=main_modality,
                        all_modalities=modalities_to_use)

    model = TopicModel(
        artm_model=model,
        custom_regularizers=custom_regs
    )
    model.has_bcg = True  # TODO: only if init_bcg_sparse_model

    return model


def init_bcg_sparse_model(
        dataset,
        modalities_to_use,
        main_modality,
        specific_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str or dict
    main_modality : str
    specific_topics : int
    bcg_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_plsa(
        dataset, modalities_to_use, main_modality, specific_topics, bcg_topics
    )
    background_topic_names = model.topic_names[-bcg_topics:]

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    dictionary = dataset.get_dictionary()
    baseline_class_ids = {class_id: 1 for class_id in modalities_to_use}
    data_stats = count_vocab_size(dictionary, baseline_class_ids)

    # all coefficients are relative
    regularizers = [
        artm.SmoothSparsePhiRegularizer(
             name='smooth_phi_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
             class_ids=[main_modality],
        ),
        artm.SmoothSparseThetaRegularizer(
             name='smooth_theta_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
        ),
        artm.SmoothSparsePhiRegularizer(
             name='sparse_phi_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
             class_ids=[main_modality],
            ),
        artm.SmoothSparseThetaRegularizer(
             name='sparse_theta_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
        ),
    ]
    for reg in regularizers:
        model.regularizers.add(transform_regularizer(
            data_stats,
            reg,
            model.class_ids,
            n_topics=len(reg.topic_names)
        ))

    return model


def init_baseline_artm(
        dataset,
        modalities_to_use,
        main_modality,
        num_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None,
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str
    main_modality : str
    num_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_bcg_sparse_model(
        dataset, modalities_to_use, main_modality, num_topics, bcg_topics, specific_topic_names, model_params
    )

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    model.regularizers.add(
        artm.DecorrelatorPhiRegularizer(
            gamma=0,
            tau=model_params.get('decorrelation_tau', 0.01),
            name='decorrelation',
            topic_names=specific_topic_names,
            class_ids=modalities_to_use,
        )
    )

    return model

In [68]:
def artm_train_func(
        dataset: Dataset,
        model_number: int,
        num_topics: int,
        num_fit_iterations: int,
        scores: List = None,
        **kwargs) -> TopicModel:
    """

    Additional Parameters
    ---------------------
    kwargs
        Some params for `_get_topic_model`, such as `cache_theta` and `num_processors`
    """

    topic_model = init_model_from_family(
        family='ARTM',
        dataset=DATASET,
        main_modality=MAIN_MODALITY,
        num_topics=ONE_MODEL_NUM_TOPICS,
        seed=model_number,
        model_params={
            'decorrelation_tau': 0.01,  # best values
            'smooth_bcg_tau': 0.05,
            'sparse_sp_tau': -0.05,
        }
    )

    num_fit_iterations_with_scores = 1

    topic_model._fit(
        dataset.get_batch_vectorizer(),
        num_iterations=max(0, num_fit_iterations - num_fit_iterations_with_scores)
    )
    _fit_model_with_scores(
        topic_model,
        DATASET,
        scores,
        num_fit_iterations=num_fit_iterations_with_scores
    )

    return topic_model


def _fit_model_with_scores(
        topic_model: TopicModel,
        dataset: Dataset,
        scores: List = None,
        num_fit_iterations: int = 1):

    if scores is not None:
        for score in scores:
            score._attach(topic_model)

    topic_model._fit(
        dataset.get_batch_vectorizer(),
        num_iterations=num_fit_iterations
    )

### Bank Creation<a id="bank-creation"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Here we finally run the experiment!

In [69]:
NUM_ITERATIONS = 20

In [70]:
seed = 0

In [71]:
# We use only one train function here
# Other variations are also possible
# It would be even better to make bank using several train functions
# However, it would also take way more time 

TRAIN_FUNCS = artm_train_func  # default train func

In [72]:
DATASET_INTERNALS_FOLDER_PATH

'./MKB_10__internals'

In [73]:
SEARCH_RESULTS_FOLDER_PATH = os.path.join(
    DATASET_INTERNALS_FOLDER_PATH, 'result2_50'
)

# File with some info about the process
SEARCH_RESULT_FILE_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'search_result__{seed}.json'
)

# Bank, with topics and their score values
BANK_FOLDER_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'bank__{seed}'
)

In [74]:
SEARCH_RESULTS_FOLDER_PATH

'./MKB_10__internals/result2_50'

In [75]:
BANK_FOLDER_PATH

'./MKB_10__internals/result2_50/bank__0'

In [76]:
os.makedirs(SEARCH_RESULTS_FOLDER_PATH, exist_ok=True)
os.makedirs(BANK_FOLDER_PATH, exist_ok=True)

In [77]:
seed

0

One cay vary some parameters below (for example `max_num_models` and `num_fit_iterations`).

In [78]:
optimizer = TopicBankMethod(
    data        = DATASET,
    main_modality = MAIN_MODALITY,
    
    min_df_rate = 0.0,  # dictionary filtering has already been done little earlier
    max_df_rate = 1.0,  #   so we don't want these parameters to have any effect

    main_topic_score   = coherence_score,
    other_topic_scores = [],
    other_scores       = [coherence_score] + diversity_scores + other_scores,
   # documents          = TEST_DOCUMENTS,

    start_model_number   = 0,
    max_num_models       = 20,
    one_model_num_topics = ONE_MODEL_NUM_TOPICS,  # 100,
    num_fit_iterations   = NUM_ITERATIONS,  # 100,  # 100 should be enough;
                                 # however, for big data better to reduce this one
                                 # (otherwise the process will be too slow)

    topic_score_threshold_percentile = 0.8997333820973618,  # DIFF ALSO HERE

    save_bank         = True,
    save_model_topics = True,
    save_file_path    = SEARCH_RESULT_FILE_PATH,
    bank_folder_path  = BANK_FOLDER_PATH,

    train_funcs = TRAIN_FUNCS,
    
    verbose = True,
)

# TODO: use Holdout Perplexity as Stop score

/home/alekseev_v/projects/iterative/../OptimalNumberOfTopics/topnum/search_methods/topic_bank/topic_bank_method.py:208: UserWarning: topic_score_threshold_percentile 0.8997333820973618 is less than one! It is expected to be in [0, 100]. Are you sure you want to proceed (yes/no)?
  warnings.warn(


In [79]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

Checking file paths

In [80]:
! ls ./Post_Science__internals

batches    _result  _result2  result2_50  result_unfiltered_dict
dict.dict  result   result2   result_50   vw.txt


In [81]:
optimizer._save_file_path

'./MKB_10__internals/result2_50/search_result__0.json'

In [82]:
optimizer._topic_bank._path

'./MKB_10__internals/result2_50/bank__0'

Fulfilling the search (get ready for a really long process!):

In [83]:
%%time

optimizer.search_for_optimum(DATASET)

  0%|                                                        | 0/20 [00:00<?, ?it/s]No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Detected bcg topics! Skipping for diversity computation (and now 50 topics).
Detected bcg topics! Skipping for diversity computation (and now 50 topics).
Detected bcg topics! Skipping for diversity computation (and now 50 topics).
Detected bcg topics! Skipping for diversity computation (and now 50 topics).

  0%|                                                         | 0/1 [00:00<?, ?it/s]Detected bcg topics! Skipping for coherence computation (and will have 50 topics).

100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.39it/s]
Using absoulte threshold: 0.8997333820973618.
Skipping saving scores for bcg topic


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 9181.0693359375, 'coherence_20': 1.1329603981100815, 'diversity_euclidean': 0.08935534144484748, 'diversity_jensenshannon': 0.6899422849567516, 'diversity_hellinger': 0.8107274888516574, 'diversity_cosine': 0.8788943166097469, 'perplexity': 9181.0693359375, 'ppl_fair': 9181.0693359375, 'ppl_cheatty': 3414.854248046875}
  5%|██▍                                             | 1/20 [01:11<22:29, 71.03s/it]No spec topics
No spec topics


...

Bank scores: {'perplexity_score': 7151.40380859375, 'coherence_20': 1.088585102325107, 'diversity_euclidean': 0.08661293091543688, 'diversity_jensenshannon': 0.6827145362875345, 'diversity_hellinger': 0.8019615375516269, 'diversity_cosine': 0.868807038674489, 'perplexity': 7151.40380859375, 'ppl_fair': 7151.40380859375, 'ppl_cheatty': 3276.736328125}
 95%|████████████████████████████████████████████▋  | 19/20 [25:37<01:22, 82.45s/it]No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Detected bcg topics! Skipping for diversity computation (and now 50 topics).
Detected bcg topics! Skipping for diversity computation (and now 50 topics).
Detected bcg topics! Skipping for diversity computation (and now 50 topics).
Detected bcg topics! Skipping for diversity computation (and now 50 topics).

  0%|                                                         | 0/1 [00:00<?, ?it/s]Detected bcg topics! Skipping for coherence computation (and will have 50 topics).

100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.16it/s]
Using absoulte threshold: 0.8997333820973618.
Creating first level with 14 topics. Dictionary: artm.Dictionary(name=a536dcf1-32ce-4d6e-93c2-0994face6039, num_entries=22608).
Copying phi for the first level. Phi shape: (22608, 14). First words: MultiIndex([('@text',              'cdk'),
            ('@text',    'внутриядерный'),
            ('@text',      'соотнесение'),
            ('@text',    'ангиоматозный'),
            ('@text', 

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 7365.70166015625, 'coherence_20': 1.0940928784569097, 'diversity_euclidean': 0.08597331388915083, 'diversity_jensenshannon': 0.6795176507276979, 'diversity_hellinger': 0.797904102369454, 'diversity_cosine': 0.8667473333046445, 'perplexity': 7365.70166015625, 'ppl_fair': 7365.70166015625, 'ppl_cheatty': 3324.31787109375}
100%|███████████████████████████████████████████████| 20/20 [27:06<00:00, 81.31s/it]
CPU times: user 42min 1s, sys: 1min, total: 43min 2s
Wall time: 27min 6s


In [95]:
optimizer._main_modality

'@text'

What topics we have in bank

In [96]:
optimizer._topic_bank.view_topics().head()

topic_0  topic_1  topic_2  topic_3  topic_4  topic_5  \
@text cdk                0.0      0.0      0.0      0.0      0.0      0.0   
      внутриядерный      0.0      0.0      0.0      0.0      0.0      0.0   
      соотнесение        0.0      0.0      0.0      0.0      0.0      0.0   
      ангиоматозный      0.0      0.0      0.0      0.0      0.0      0.0   
      хориоменингит      0.0      0.0      0.0      0.0      0.0      0.0   

                     topic_6  topic_7  topic_8  topic_9  topic_10  topic_11  \
@text cdk                0.0      0.0      0.0      0.0       0.0       0.0   
      внутриядерный      0.0      0.0      0.0      0.0       0.0       0.0   
      соотнесение        0.0      0.0      0.0      0.0       0.0       0.0   
      ангиоматозный      0.0      0.0      0.0      0.0       0.0       0.0   
      хориоменингит      0.0      0.0      0.0      0.0       0.0       0.0   

                     topic_12  topic_13  
@text cdk                 0.0       0.0  
      внутриядерный       0.0       0.0  
      соотнесение         0.0       0.0  
      ангиоматозный       0.0       0.0  
      хориоменингит       0.0       0.0

In [97]:
bank_topics = optimizer._topic_bank.view_topics()

In [98]:
bank_topics.shape

(22608, 14)

In [99]:
bank_topics.head()

topic_0  topic_1  topic_2  topic_3  topic_4  topic_5  \
@text cdk                0.0      0.0      0.0      0.0      0.0      0.0   
      внутриядерный      0.0      0.0      0.0      0.0      0.0      0.0   
      соотнесение        0.0      0.0      0.0      0.0      0.0      0.0   
      ангиоматозный      0.0      0.0      0.0      0.0      0.0      0.0   
      хориоменингит      0.0      0.0      0.0      0.0      0.0      0.0   

                     topic_6  topic_7  topic_8  topic_9  topic_10  topic_11  \
@text cdk                0.0      0.0      0.0      0.0       0.0       0.0   
      внутриядерный      0.0      0.0      0.0      0.0       0.0       0.0   
      соотнесение        0.0      0.0      0.0      0.0       0.0       0.0   
      ангиоматозный      0.0      0.0      0.0      0.0       0.0       0.0   
      хориоменингит      0.0      0.0      0.0      0.0       0.0       0.0   

                     topic_12  topic_13  
@text cdk                 0.0       0.0  
      внутриядерный       0.0       0.0  
      соотнесение         0.0       0.0  
      ангиоматозный       0.0       0.0  
      хориоменингит       0.0       0.0

In [103]:
bank_topics['topic_13'].sort_values(ascending=False)[:20]

@text  позвоночник       0.018917
       грыжа             0.012712
       диск              0.011714
       метод             0.011429
       операция          0.009809
       стопа             0.008426
       палец             0.007709
       позвонок          0.007207
       деформация        0.006755
       позвоночный       0.006755
       изменение         0.006467
       канал             0.006263
       межпозвонковый    0.005290
       ткань             0.004930
       хирургический     0.004904
       отдел             0.004802
       нагрузка          0.004668
       облучение         0.004560
       искривление       0.004551
       процесс           0.004455
Name: topic_13, dtype: float64

And topic scores

In [104]:
optimizer._topic_bank.view_topic_scores()

,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,topic_10,topic_11,topic_12,topic_13
kernel_size,2154.000000,1925.000000,2330.000000,1745.000000,1882.000000,1964.000000,2067.000000,1722.000000,2626.000000,2594.000000,2347.000000,2184.000000,2159.000000,2091.000000
coherence_20,1.113181,1.199071,1.008168,1.581345,1.030367,0.900842,1.254779,0.969312,0.956957,1.096151,0.941750,1.077395,1.222141,0.965841
distance_to_nearest,0.000000,0.912625,0.790161,0.862318,0.817064,0.830169,0.831680,0.653847,0.823542,0.733325,0.827345,0.832594,0.831554,0.768941


All models are also saved (topics as $\Phi$ matrices and topic score values)

In [105]:
! ls $optimizer._topic_bank._path

model_0__phi.bin	    model_19__topic_scores.bin
model_0__topic_scores.bin   model_1__phi.bin
model_10__phi.bin	    model_1__topic_scores.bin
model_10__topic_scores.bin  model_2__phi.bin
model_11__phi.bin	    model_2__topic_scores.bin
model_11__topic_scores.bin  model_3__phi.bin
model_12__phi.bin	    model_3__topic_scores.bin
model_12__topic_scores.bin  model_4__phi.bin
model_13__phi.bin	    model_4__topic_scores.bin
model_13__topic_scores.bin  model_5__phi.bin
model_14__phi.bin	    model_5__topic_scores.bin
model_14__topic_scores.bin  model_6__phi.bin
model_15__phi.bin	    model_6__topic_scores.bin
model_15__topic_scores.bin  model_7__phi.bin
model_16__phi.bin	    model_7__topic_scores.bin
model_16__topic_scores.bin  model_8__phi.bin
model_17__phi.bin	    model_8__topic_scores.bin
model_17__topic_scores.bin  model_9__phi.bin
model_18__phi.bin	    model_9__topic_scores.bin
model_18__topic_scores.bin  topics.bin
model_19__phi.bin	    topic_scores.bin


In [106]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

In [107]:
optimizer._result['num_bank_topics']

[11,
 11,
 12,
 12,
 13,
 14,
 14,
 14,
 14,
 14,
 14,
 14,
 14,
 14,
 14,
 14,
 14,
 14,
 14,
 14]

In [108]:
len(optimizer._result['bank_topic_scores'])

20

In [111]:
optimizer._result['bank_scores'][-1]

{'perplexity_score': 7365.70166015625,
 'coherence_20': 1.0940928784569097,
 'diversity_euclidean': 0.08597331388915083,
 'diversity_jensenshannon': 0.6795176507276979,
 'diversity_hellinger': 0.797904102369454,
 'diversity_cosine': 0.8667473333046445,
 'perplexity': 7365.70166015625,
 'ppl_fair': 7365.70166015625,
 'ppl_cheatty': 3324.31787109375}

In [112]:
len(optimizer._result['bank_scores'])

20

In [113]:
! echo $SEARCH_RESULTS_FOLDER_PATH
! ls $SEARCH_RESULTS_FOLDER_PATH

./MKB_10__internals/result2_50
bank__0  search_result__0.json
